# Tutorial: Simulating ancient navigation with Pytheas Simulator

> Author: Matteo Tomasini
>
> Updated: 12.10.2026

In this tutorial, I will go through the steps that are needed to get Python Simulator to work on your laptop, as well as running a small simulation example. In this Python Notebook I will go through the important steps, but there is also a script to run once you have downloaded the data and set up the simulator. 

The aim of the tutorial is for you to familiarize yourself with Pytheas Simulator, by simulating a crossing and a coastal travel in the Kattegat strait. The tutorial will also show you how to download data for one year (to illustrate how the data folders are structured).

Let's get going! 

## The data

First, we need to download the environmental data necessary for Pytheas to function: currents, winds, and waves. The data need to be structured in a specific format. From your data folder, the structure is:

```
.
├── currents
│   └── 2025
│       └── month_1.nc
|       └── month_2.nc
|       └── ...
|       └── month_12.nc
├── waves
│   └── 2025
│       └── month_1.nc
|       └── month_2.nc
|       └── ...
|       └── month_12.nc
└── winds
        └── 2025
            └── month_1.nc
            └── month_2.nc
            └── ...
            └── month_12.nc
```

If you have downloaded this tutorial and the corresponding folders from https://github.com/mtomasini/PytheasSimulator/, the data folder is `./data/environmental_data/`.

### Copernicus data stores

To download the data from the Copernicus Data Store, you first need to obtain a login for the [ECMWF Climate Data Store](https://cds.climate.copernicus.eu/) and the [Copernicus Marine Data Store](https://data.marine.copernicus.eu/) -- in both cases, click on "Login" and follow the indications. Once you have obtained these, you can go through with the rest of the tutorial.

Some additional set-up is needed for the provided script to work. The process is different for different operating systems, please refer to the official documentation to find your set-up:

- ECMWF Data Store: https://cds.climate.copernicus.eu/how-to-api
- Copernicus Marine Data Store: https://help.marine.copernicus.eu/en/articles/7949409-copernicus-marine-toolbox-introduction

### Winds download

The winds are downloaded from the [CERRA sub-daily regional reanalysis data for Europe on single levels from 1984 to present](https://cds.climate.copernicus.eu/datasets/reanalysis-cerra-single-levels?tab=overview); in the specific the script downloads the variables `10m wind direction` and `10m wind speed`. Be sure to have set up your profile and your API, and then run:

```python
import cdsapi

months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

for month in months:

    dataset = "reanalysis-cerra-single-levels"
    request = {
        "variable": [
            "10m_wind_direction",
            "10m_wind_speed"
        ],
        "level_type": "surface_or_atmosphere",
        "data_type": ["reanalysis"],
        "product_type": "analysis",
        "year": ["1995"],
        "month": [
            f"{month}"
        ],
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "time": [
            "00:00", "06:00",
            "12:00", "18:00"
        ],
        "data_format": "netcdf"
    }

    client = cdsapi.Client()
    target = f"./data/environmental_data/winds/2025/month_{month}.nc"
    client.retrieve(dataset, request, target).download()

```

ATTENTION! Do not run this code in this tutorial. It will send a request to CDS to download your data, but will put you in a queue, especially in times of high traffic. This may take a while to run! The same code can be found in the script `./data/import_winds.py`.

### Currents and waves

Currents data and waves data are downloaded from the same source. Here too, refer to the documentation to set-up your login and your API requests. For the Arctic, the datasets we are interested in are the [Arctic Ocean Physics Reanalysis](https://data.marine.copernicus.eu/product/ARCTIC_MULTIYEAR_PHY_002_003/download) (`cmems_mod_arc_phy_my_topaz4_P1D-m`) for the currents, and the waves dataset is the [Arctic Ocean Waves Hindcast](https://data.marine.copernicus.eu/product/ARCTIC_MULTIYEAR_WAV_002_013/download) (`cmems_mod_arc_wav_my_3km-climatology_P1M-m`).


Then, you can run the following script making use of the Copernicus Marine Toolbox. This will ask for your login credentials before it proceeds to downloading your two datasets.

```python
import copernicusmarine

def monthly_import(dataset, type_of_dataset):
    months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

    if type_of_dataset == "currents":
        # ATTENTION! there are datasets out there that call currents in the x and y component "uo" and "vo" instead!
        variables == ["vxo", "vyo"]
    elif type_of_dataset == "waves":
        variables == ["VHM0"]

    for month in months:
        last_day = "31" 
        if month in ["04", "06", "09", "11"]: 
            last_day = "30"
        elif month == "02":
            # here we are downloading for 2025, but in case of a leap year this should be "29"!
            last_day = "28"
        copernicusmarine.subset(
            dataset_id=dataset,
            variables=variables,
            minimum_longitude=-29.720039021480375,
            maximum_longitude=10.480836622997344,
            minimum_latitude=55.8011102398125,
            maximum_latitude=65.84004107063826,
            start_datetime=f"1995-{month}-01T00:00:00",
            end_datetime=f"1995-{month}-{last_day}T23:59:59",
            minimum_depth=0,
            maximum_depth=0,
            output_filename = f"month_{month}.nc",
            output_directory = f"./data/environmental_data/{type_of_dataset}/"
    )

monthly_import("cmems_mod_arc_phy_my_topaz4_P1D-m", "currents")
monthly_import("cmems_mod_arc_wav_my_3km-climatology_P1M-m", "waves")
```
Again, this is exactly the code you find in the script `./data/import_currents-waves.py`.

### Pre-shipped dataset

For confort of usage, I include in the data shipped with this tutorial a month of data for the Baltic for 1996. The wind data comes from the same dataset specified above (CERRA contains data for the whole of Europe, including part of the Arctic and the Mediterranean Sea); the currents come from the dataset [`cmemes_mod_bal_phy_my_P1D-m`](https://data.marine.copernicus.eu/product/BALTICSEA_MULTIYEAR_PHY_003_011/download); the waves from the dataset [`cmems_mod_bal_wav_my_PT1H-i`](https://data.marine.copernicus.eu/product/BALTICSEA_MULTIYEAR_WAV_003_015/download).

## Simulation

Let us simulate two voyages: one is a direct crossing between Gothenburg and the northernmost tip of Denmark, Skagen. The second voyage will be a coastal voyage from Gothenburg northward to Tanumshede. Let's start by defining the locations:

In [ ]:
GOTHENBURG = [57.66, 11.72]
SKAGEN = [57.72, 10.61]
TANUM = [58.70, 11.16]

We then define some variables an import the necessary packages.

The polar diagrams

In [ ]:
polar_diagarms_path = "hjortspring_speeds_16pad_3000kg_44cad_75oars.txt"